# Risk Metrics

This notebook creates separate downloadable CSVs for the supported **Population**, **Capital Stock**, and **Direct Damage** Risk cards. Every file identifies its Risk subsection and analytical source. Population and Capital Stock are limited to river flooding; Direct Damage retains its supported hazards as rows.

## 0. Setup

Country settings, administrative level, source paths, and model-run bundles come from the selected country configuration. Shared calculations live in `src/national_tool_metrics/sections/risk.py`.

In [ ]:
from pathlib import Path
import importlib
import sys

WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
SRC_DIRECTORY = REPO_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from national_tool_metrics import load_country_config
from national_tool_metrics.boundaries import load_admin_boundaries
from national_tool_metrics.outputs import write_card_output
import national_tool_metrics.sections.risk as risk_section

importlib.reload(risk_section)
from national_tool_metrics.sections.risk import (
    RISK_CARD_DIMENSIONS,
    RISK_CARD_OPTIONAL_DIMENSIONS,
    assemble_risk_card_metrics,
    build_capital_stock_risk_metrics,
    build_direct_network_risk_metrics,
    build_population_risk_metrics,
)

In [ ]:
config = load_country_config("MOZ", repo_root=REPO_ROOT)
admin_regions = load_admin_boundaries(config)

river_run = config.risk_run("river_flood_jrc_baseline")
cyclone_run = config.risk_run("tropical_cyclone_storm_baseline_2020")

print(f"Country: {config.country.name} ({config.country.iso3})")
print(f"Administrative level: {config.country.admin_level.upper()}")
print(f"Administrative regions: {len(admin_regions):,}")
print(f"Risk runs: {river_run.name}, {cyclone_run.name}")

## 1. Socioeconomic Risk — Population

Load the JRC protection-adjusted expected annual flooded population (`AAR_protected`) and the `RP10`, `RP20`, `RP50`, `RP75`, `RP100`, `RP200`, and `RP500` event exposure metrics for the eight demographic groups and five national wealth quintiles. The risk map is encoded in each metric column name; unprotected AAR rows are intentionally excluded.

In [ ]:
population_risk_metrics = build_population_risk_metrics(
    config,
    admin_regions,
    river_run,
)
population_risk_metrics.head()

## 2. Socioeconomic Risk — Capital Stock

Load precomputed JRC protection-adjusted average annual losses and `RP10`, `RP20`, `RP50`, `RP75`, `RP100`, `RP200`, and `RP500` event losses for residential, non-residential, infrastructure, and total capital stock. The risk map is encoded in each metric column name.

In [ ]:
capital_stock_risk_metrics = build_capital_stock_risk_metrics(
    config,
    admin_regions,
    river_run,
)
capital_stock_risk_metrics.head()

## 3. Direct Infrastructure Risk — Roads and Rail

Allocate JRC river-flood expected annual damage to administrative regions by intersected network length. Road EAD is reported as a total and by road class; rail is reported as a total.

In [ ]:
river_direct_risk_metrics = build_direct_network_risk_metrics(
    config,
    admin_regions,
    river_run,
)
river_direct_risk_metrics.head()

## 4. Direct Infrastructure Risk — Power

Allocate STORM tropical-cyclone power-network EAD to administrative regions. An all-zero result is retained as valid for Kenya; the same workflow can be reused in countries with greater tropical-cyclone exposure.

In [ ]:
cyclone_direct_risk_metrics = build_direct_network_risk_metrics(
    config,
    admin_regions,
    cyclone_run,
)
cyclone_direct_risk_metrics.head()

## 5. Deferred Risk Components

**Indirect Impacts**, **Facilities**, and **Accessibility** are not calculated or exported here. They can later adopt the same one-CSV-per-card structure without changing these three outputs.

## 6. Assemble Card Tables

Reshape the existing calculations into tidy card tables. Population and Capital Stock use one `risk_metric` column for average-annual and return-period results. Direct Damage keeps `hazard` and optional `epoch` dimensions for its multi-hazard rows.

In [ ]:
risk_card_metrics = assemble_risk_card_metrics(
    config,
    admin_regions,
    population_risk_metrics,
    capital_stock_risk_metrics,
    river_direct_risk_metrics,
    cyclone_direct_risk_metrics,
)

for card, metrics in risk_card_metrics.items():
    print(f"{card}: {len(metrics):,} rows")

risk_card_metrics["population"].head()

## 7. Export

Write one downloadable CSV for each supported Risk card.

In [ ]:
output_paths = {}
for card, metrics in risk_card_metrics.items():
    output_paths[card] = write_card_output(
        metrics,
        config,
        section="risk",
        card=card,
        dimension_columns=RISK_CARD_DIMENSIONS[card],
        optional_dimension_columns=RISK_CARD_OPTIONAL_DIMENSIONS[card],
    )
    print(f"Exported {card} metrics to: {output_paths[card]}")